In [1]:
import math
import numpy as np
from gensim.models import LdaModel
from gensim.corpora import Dictionary

In [ ]:
def bow(doc):
    words = doc.split(' ')[1:]
    words = [w.split(':') for w in words]
    words = {vocab[int(w[0])]: int(w[1].strip('\n')) for w in words}
    return words

data = []
corpus_features_path = 'path_to_corpus_features/'
with open(f'{corpus_features_path}/train.feat', 'r') as f:
    data += f.readlines()
with open(f'{corpus_features_path}/test.feat', 'r') as f:
    data += f.readlines()
with open(f'{corpus_features_path}/vocab', 'r') as f:
    vocab = f.readlines()
vocab = [w.split(' ') for w in vocab]
vocab = {int(w[1].strip('\n')): w[0] for w in vocab}
bow_data = [bow(doc) for doc in data]
D = len(bow_data)

V = len(vocab)
occurence_sets = {vocab[i]: set({}) for i in range(1,V+1)}
for i, doc in enumerate(bow_data):
    for k in doc.keys():
        occurence_sets[k].add(i)

def occurences(w, p=False):
    c = len(occurence_sets[w])
    return c / D if p else c

def co_occurences(w1, w2, p=False):
    c1 = occurence_sets[w1]
    c2 = occurence_sets[w2]
    c = len(c1.intersection(c2))
    return c / D if p else c

In [3]:
def umass(topic):
    score = 0
    K = len(topic)
    for i in range(K):
        for j in range(i+1,K):
            w1 = topic[i]
            w2 = topic[j]
            p1 = occurences(w2, p=True)
            p12 = co_occurences(w1, w2, p=True)
            if p12 < p1:
                score += math.log((p12 + 1/D) / p1, 10)
    score /= K * (K-1) // 2
    return score

def lcp(topic):
    score = 0
    K = len(topic)
    for i in range(K):
        for j in range(i+1,K):
            w1 = topic[i]
            w2 = topic[j]
            p1 = occurences(w1, p=True)
            p2 = occurences(w2, p=True)
            p12 = co_occurences(w1, w2, p=True)
            if p12 > 0:
                score += math.log(p12 / p1, 10)
            else:
                score += math.log(p2, 10)
    score /= K * (K-1) // 2
    return score

def pmi(topic):
    score = 0
    K = len(topic)
    for i in range(K):
        for j in range(i+1,K):
            w1 = topic[i]
            w2 = topic[j]
            p1 = occurences(w1, p=True)
            p2 = occurences(w2, p=True)
            p12 = co_occurences(w1, w2, p=True)
            if p12 > 0:
                score += math.log(p12 / (p1 * p2), 10)
    score /= K * (K-1) // 2
    return score

def npmi(topic):
    score = 0
    K = len(topic)
    for i in range(K):
        for j in range(i+1,K):
            w1 = topic[i]
            w2 = topic[j]
            p1 = occurences(w1, p=True)
            p2 = occurences(w2, p=True)
            p12 = co_occurences(w1, w2, p=True)
            if p12 > 0:
                pmi = math.log(p12 / (p1 * p2), 10)
                score += -pmi / math.log(p12, 10)
    score /= K * (K-1) // 2
    return score

def topic_coherence(topics, func):
    scores = [func(topic) for topic in topics]
    score = sum(scores) / len(scores)
    return (score, max(scores))

In [ ]:
def parse(doc):
    words = doc.split(' ')
    words[-1] = words[-1].strip('\n')
    return words

documents = []
corpus_features_path = 'path_to_corpus_features/'
with open(f'{corpus_features_path}/train.txt', 'r') as f:
    documents += f.readlines()
with open(f'{corpus_features_path}/test.txt', 'r') as f:
    documents += f.readlines()
documents = [parse(doc) for doc in documents]
dictionary = Dictionary(documents)
corpus = [dictionary.doc2bow(doc) for doc in documents]

In [ ]:
def parse(doc):
    words = doc.split(' ')
    words[-1] = words[-1].strip('\n')
    return words

K = 10
lda = np.zeros((4,5))
for seed in range(42, 47):
    documents = []
    corpus_features_path = f'path_to_corpus_features/ap-seed={seed}'
    print(f'Training LDA seed = {seed}')
    with open(f'{corpus_features_path}/train.txt', 'r') as f:
        documents += f.readlines()
    with open(f'{corpus_features_path}/test.txt', 'r') as f:
        documents += f.readlines()
    documents = [parse(doc) for doc in documents]
    dictionary = Dictionary(documents)
    corpus = [dictionary.doc2bow(doc) for doc in documents]
    lda_model = LdaModel(
        corpus=corpus,
        id2word=dictionary,
        num_topics=100,
        passes=100,
    )
    lda_topics = []
    for topic_id in range(lda_model.num_topics):
        words = lda_model.show_topic(topic_id, topn=K)
        lda_topics.append([word for word, _ in words])
    lda_topics = [' '.join(topic) for topic in lda_topics]
    with open(f'./topic-100-seed={seed}.txt', 'w') as f:
        for item in lda_topics:
            f.write(item + '\n')

Training LDA seed = 42
Training LDA seed = 43
Training LDA seed = 44
Training LDA seed = 45
Training LDA seed = 46
